# In-Context Learning


In-context learning is a generalisation of few-shot learning where the LLM is provided a context as part of the prompt and asked to respond by utilising the information in the context.

* Example: *"Summarize this research article into one paragraph highlighting its strengths and weaknesses: [insert article text]”*
* Example: *"Extract all the quotes from this text and organize them in alphabetical order: [insert text]”*

A very popular technique that you will learn in week 5 called Retrieval-Augmented Generation (RAG) is a form of in-context learning, where:
* a search engine is used to retrieve some relevant information
* that information is then provided to the LLM as context


In this example we download some recent research papers from arXiv papers, extract the text from the PDF files and ask Gemini to summarize the articles as well as provide the main strengths and weaknesses of the papers. Finally we print the summaries to a local html file and as markdown.

In [2]:
!pip install requests bs4 google-generativeai pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 7.3 MB/s eta 0:00:00


In [20]:
!pip install markdown

In [3]:
import os
import requests
from bs4 import BeautifulSoup
import google.generativeai as genai
from urllib.request import urlopen, urlretrieve
from IPython.display import Markdown, display
from pypdf import PdfReader
from datetime import date
from tqdm import tqdm
from google.colab import userdata

In [4]:
import google.generativeai as genai
from google.colab import userdata

API_KEY = userdata.get("helsinki")
if not API_KEY:
    raise ValueError("No API key found in Colab Secrets (expected secret named 'helsinki').")

genai.configure(api_key=API_KEY)

We select those papers that have been featured in Hugging Face papers.

In [5]:
BASE_URL = "https://huggingface.co/papers"
page = requests.get(BASE_URL)
soup = BeautifulSoup(page.content, "html.parser")
h3s = soup.find_all("h3")

papers = []

for h3 in h3s:
    a = h3.find("a")
    title = a.text
    link = a["href"].replace('/papers', '')

    papers.append({"title": title, "url": f"https://arxiv.org/pdf{link}"})

Code to extract text from PDFs.

In [6]:
def extract_paper(url):
    html = urlopen(url).read()
    soup = BeautifulSoup(html, features="html.parser")

    # kill all script and style elements
    for script in soup(["script", "style"]):
        script.extract()    # rip it out

    # get text
    text = soup.get_text()

    # break into lines and remove leading and trailing space on each
    lines = (line.strip() for line in text.splitlines())
    # break multi-headlines into a line each
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    # drop blank lines
    text = '\n'.join(chunk for chunk in chunks if chunk)

    return text


def extract_pdf(url):
    pdf = urlretrieve(url, "pdf_file.pdf")
    reader = PdfReader("pdf_file.pdf")
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text


def printmd(string):
    display(Markdown(string))

In [7]:
LLM = "gemini-2.5-flash"
model = genai.GenerativeModel(LLM)

We use Gemini to summarize the papers.

In [23]:
def strengths_weaknesses_prompt(title, paper_text):
    return f"""
You are reviewing a research paper.

Return ONLY a Markdown table with exactly 2 columns (Strengths, Weaknesses) and exactly 1 content row.

Use this exact format (replace the placeholders, keep the pipes):

| Strengths | Weaknesses |
|-----------|------------|
| - s1<br>- s2<br>- s3 | - w1<br>- w2<br>- w3 |

Rules:
- 3–6 bullets per cell
- Bullets must stay INSIDE the single table row (no bullets outside the table)
- No extra text before/after the table

Paper title (do not repeat it): {title}

Paper text:
{paper_text[:20000]}
""".strip()


We print the results to a html file.

In [24]:
import markdown
from datetime import date

style = """
<style>
  body { font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial, sans-serif; max-width: 980px; margin: 24px auto; padding: 0 16px; line-height: 1.4; }
  h1 { margin: 0 0 6px; }
  h4 { margin: 0 0 18px; color: #555; font-weight: 500; }
  h2 { margin-top: 28px; }
  table { border-collapse: collapse; width: 100%; margin: 10px 0 18px; }
  th, td { border: 1px solid #ddd; padding: 10px 12px; vertical-align: top; }
  th { background: #f6f6f6; text-align: left; }
  td { background: #fff; }
  a { color: #1a73e8; text-decoration: none; }
  a:hover { text-decoration: underline; }
</style>
"""

page = f"<html><head><meta charset='utf-8'>{style}</head><body><h1>Daily Dose of AI Research</h1><h4>{date.today()}</h4><p><i>Summaries generated with: {LLM}</i></p>"
with open("papers.html", "w", encoding="utf-8") as f:
    f.write(page)

for paper in papers:
    html_table = markdown.markdown(paper["summary"], extensions=["tables"])
    page = f'<h2><a href="{paper["url"]}">{paper["title"]}</a></h2>{html_table}'
    with open("papers.html", "a", encoding="utf-8") as f:
        f.write(page)

end = "</body></html>"
with open("papers.html", "a", encoding="utf-8") as f:
    f.write(end)

We can also print the results to this notebook as markdown.

In [25]:
for paper in papers:
    printmd("**[{}]({})**<br>{}<br><br>".format(paper["title"],
                                                paper["url"],
                                                paper["summary"]))


**[Latent Implicit Visual Reasoning](https://arxiv.org/pdf/2512.21218)**<br>| Strengths / Weaknesses |
|---|---|
| - Proposes a novel task-agnostic mechanism for LMMs to implicitly learn visual reasoning tokens without explicit supervision.<br>- Consistently outperforms direct supervised fine-tuning (SFT) across a diverse range of perception-heavy tasks and multiple LMM backbones.<br>- Demonstrates strong generalization capabilities, improving performance in both single-task and multi-task fine-tuning settings.<br>- Avoids the need for additional annotated data or human-designed intermediate visual steps, reducing annotation costs and biases.<br>- Comprehensive ablation studies and visualizations support the method's design and demonstrate learned visual information. | - Latent tokens inherently offer less interpretability compared to textual explanations, making it harder to fully understand the exact reasoning steps.<br>- Performance can be sensitive to specific hyperparameters, such as the Stage-1/Stage-2 training schedule and the number of latent tokens (K).<br>- Freezing the vision encoder and projector might limit the full adaptation of visual feature extraction to the new reasoning approach.<br>- Evaluation primarily focuses on perception-heavy tasks from a specific benchmark, leaving broader applicability to other complex visual reasoning domains unexplored.<br>- The method's scalability and performance with much larger, more recent state-of-the-art LMMs are not investigated. |<br><br>

**[Emergent temporal abstractions in autoregressive models enable hierarchical reinforcement learning](https://arxiv.org/pdf/2512.20605)**<br>| Strengths | Weaknesses |
|---|---|
| - Introduces a novel "internal RL" paradigm that effectively overcomes inefficient token-by-token exploration.<br>- Metacontroller discovers temporally-abstract actions and their sequencing in an unsupervised manner.<br>- Demonstrates strong empirical performance, outperforming standard RL and hierarchical baselines on sparse-reward tasks.<br>- Provides mechanistic interpretability, showing emergent, linearly controllable abstract action representations.<br>- Highlights the critical role of pretraining and freezing the base autoregressive model for abstraction discovery. | - The entire approach is tightly coupled with autoregressive models, which may limit broader applicability to other architectures.<br>- The proposed architecture, involving a metacontroller with a non-causal encoder and switching unit, is complex.<br>- Relies on specific hyperparameters (e.g., `α`, `β_threshold`) whose sensitivity and tuning are not extensively discussed.<br>- Evaluation is limited to grid world and MuJoCo navigation tasks; scalability to real-world, diverse domains is not explored.<br>- The transition from a non-causal metacontroller during self-supervised training to a causal RL policy could benefit from more detailed explanation. |<br><br>

**[Spatia: Video Generation with Updatable Spatial Memory](https://arxiv.org/pdf/2512.15716)**<br>| Strengths | Weaknesses |
|---|---|
| - Introduces novel explicit 3D spatial memory (point cloud) for enhanced long-term consistency in video generation.<br>- Achieves dynamic-static disentanglement, allowing dynamic elements in static scenes.<br>- Provides explicit and geometrically grounded camera control and 3D-aware interactive editing capabilities.<br>- Demonstrates superior spatial consistency and visual quality in long-horizon generation through comprehensive evaluation.<br>- Outperforms state-of-the-art models on key benchmarks (WorldScore, RealEstate) for spatial memory and video quality.<br>- Robustly maintains scene coherence and structural integrity even when revisiting locations over extended sequences. | - Relies on external visual SLAM (MapAnything) for point cloud estimation and updates, potentially inheriting its limitations or inaccuracies.<br>- Maintaining and processing dense 3D point clouds for complex scenes may incur high computational and memory costs.<br>- Potential for accumulation of errors in the spatial memory (point cloud) over very long iterative generation sequences.<br>- Limited detailed quantitative evaluation or discussion on the quality and consistency of the *dynamic entities* themselves.<br>- The paper does not discuss the latency of the iterative generation and SLAM update process for interactive applications.<br>- Generalization to entirely novel scenes may be constrained by the training data used for both point cloud and video generation. |<br><br>

**[Schoenfeld's Anatomy of Mathematical Reasoning by Language Models](https://arxiv.org/pdf/2512.19995)**<br>| Strengths | Weaknesses |
|---|---|
|- Introduces ThinkARM, a novel, theory-grounded framework for explicit, functional reasoning step abstraction.<br>- Conducts extensive empirical analysis across 15 diverse LLMs on a large corpus of mathematical problems.<br>- Establishes a robust, scalable automated annotation pipeline leveraging GPT-5 with human-verified gold standards.<br>- Reveals consistent "cognitive heartbeat" and distinct structural differences in reasoning dynamics.<br>- Provides practical utility through diagnostic case studies on correctness and efficiency, going beyond surface metrics.<br>- Strong grounding in Schoenfeld's Episode Theory offers an interpretable, intermediate-scale analytical lens. | - Reliance on an automatic annotator (GPT-5) introduces potential labeling noise, despite high agreement.<br>- The scope is primarily limited to mathematical problem solving, restricting generalizability to other domains.<br>- The introduction of an "Answer" episode modifies Schoenfeld's original taxonomy, potentially affecting direct theoretical comparability.<br>- Sentence-level annotation, while scalable, might sacrifice finer-grained insights available from hierarchical schemes.<br>- Full episode-level analysis is limited for proprietary models where only final answers, not full traces, are observable.<br>- Requires future work to extend episode-level analysis to diverse domains, indicating current limitations in scope. |<br><br>

**[How Much 3D Do Video Foundation Models Encode?](https://arxiv.org/pdf/2512.19949)**<br>| Strengths | Weaknesses |
|---|---|
| - Proposes the first model-agnostic framework for systematically quantifying 3D awareness in Video Foundation Models (VidFMs).<br>- Provides novel and meaningful findings on 3D awareness across multiple axes (extent, factor, localization, implication).<br>- Conducts a comprehensive benchmark comparing state-of-the-art VidFMs, self-supervised encoders, and 3D experts.<br>- Demonstrates practical utility by showing VidFM features can significantly outperform DINO features for 3D reconstruction under limited 3D data.<br>- Employs direct 3D prediction tasks (points, depth, camera poses) rather than indirect 2.5D or optimization-based proxies.<br>- Supports findings with both extensive quantitative metrics and clear qualitative analyses. | - Relies on publicly released checkpoints, limiting controlled attribution of 3D-awareness differences to specific factors (e.g., data, training strategy).<br>- Unable to isolate the effect of data scale due to the lack of open-source models with controlled data variations.<br>- Resource constraints prevented training large-scale 3D reconstruction models from scratch with VidFM features on massive datasets.<br>- Findings show that 3D-aware fine-tuning may improve in-domain awareness but hurt generalization to other data domains.<br>- Qualitative analysis reveals that most failure cases concentrate around object boundaries, indicating a current limitation.<br>- The study's conclusions are context-dependent on specific models and probe architecture used. |<br><br>

**[VA-π: Variational Policy Alignment for Pixel-Aware Autoregressive Generation](https://arxiv.org/pdf/2512.19680)**<br>| Strengths | Weaknesses |
|---|---|
| - Addresses AR generator-tokenizer misalignment with a principled variational objective.<br>- Achieves significant performance improvements (e.g., FID, IS) on C2I and T2I generation tasks.<br>- Highly compute- and data-efficient, requiring minimal resources and no external reward models.<br>- Generalizes effectively across various AR models and unified multimodal architectures.<br>- Novel RL formulation uses intrinsic pixel-space reconstruction as reward, improving stability. | - Performance improvements are limited by the quality of the frozen, pre-trained tokenizer.<br>- Requires careful hyperparameter tuning for regularization strength (β) and contextual noise ratio (ξ).<br>- The use of teacher-forcing in training may not fully eliminate exposure bias during inference.<br>- Comparisons are primarily against other AR models, not broader state-of-the-art generative models.<br>- Optimizing the ELBO is inherently complex due to non-differentiable discrete operations. |<br><br>

**[GTR-Turbo: Merged Checkpoint is Secretly a Free Teacher for Agentic VLM Training](https://arxiv.org/pdf/2512.13043)**<br>/ Strengths / Weaknesses /
/---/---/
/ - Introduces a novel "free teacher" approach by merging checkpoints, eliminating external model dependency.<br>- Achieves significant reductions in training time (50%) and compute cost (60%) compared to prior methods.<br>- Demonstrates comparable or superior performance on complex visual agentic tasks.<br>- Effectively mitigates "entropy collapse" and ensures stable training for VLM agents.<br>- Offers flexibility with SFT and KL-regularized guidance, enhancing adaptability.<br>- Conducts comprehensive ablation studies, validating design choices and providing insights. / - Requires the base model to have a certain initial capability to avoid passive exploration issues.<br>- Experimental validation is primarily on 7B models, limiting generalizability to larger VLM scales.<br>- Cost estimations are subject to market fluctuations and commercial model economies of scale, impacting precision.<br>- Acknowledges an "unfair comparison setting" in ALFWorld due to GTR-Turbo's lack of external expert knowledge.<br>- The TIES merging method, while effective, adds complexity compared to simpler merging strategies.<br>- Optimal performance requires careful tuning of merging weights (e.g., EMA alpha) and KL estimation methods. /<br><br>

In [12]:
import os
print(os.getcwd())

/content
